In [ ]:
# ==============================================================================
# 1. AUTOMATYCZNE ODŚWIEŻANIE MODUŁÓW (BEZPIECZNE DLA PYTHON 3.12+)
# ==============================================================================
try:
    # Standardowe ładowanie, jeśli IPython jest zaktualizowany
    %load_ext autoreload
    %autoreload 2
except Exception:
    # Alternatywne obejście, jeśli środowisko zgłasza brak modułu 'imp'
    import ipykernel
    print("⚠️ Standard autoreload failed, applying compatibility patch for Python 3.12+...")
    # W nowym Pythonie używa się importlib zamiast imp
    import sys
    import types
    from importlib import reload
    # Wstrzykujemy 'imp' do sys.modules w locie, żeby oszukać stare skrypty IPythona
    imp_mock = types.ModuleType('imp')
    imp_mock.reload = reload
    sys.modules['imp'] = imp_mock
    
    # Próbujemy załadować ponownie
    %load_ext autoreload
    %autoreload 2

import os
import sys

# ==============================================================================
# 2. DEFINICJA REPOZYTORIUM GITHUB (DLA SESJI W CHMURZE COLABA)
# ==============================================================================
GITHUB_USER = "rgrocki" 
REPO_NAME = "projekt_analiza_obrazu"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Sprawdzanie: chmura Google Colab czy lokalny workspace w VS Code
IS_COLAB_CLOUD = os.path.exists("/content")

if IS_COLAB_CLOUD:
    print("🚀 [STATUS] Detected Google Colab Cloud Environment.")
    # Klonowanie repozytorium lub pobieranie najnowszych zmian (git pull)
    if not os.path.exists(REPO_NAME):
        print(f"-> Cloning repository: {REPO_NAME}...")
        !git clone {REPO_URL}
    else:
        print(f"-> Repository already exists. Pulling latest updates from GitHub...")
        !cd {REPO_NAME} && git stash && git pull
        
    # Dodanie folderu projektu do ścieżki wyszukiwania Pythona w chmurze
    path_to_append = os.path.abspath(REPO_NAME)
    if path_to_append not in sys.path:
        sys.path.append(path_to_append)
else:
    print("💻 [STATUS] Detected Local VS Code Environment.")
    print("-> Using local files from your workspace. Skipping git clone/pull.")
    
    # W VS Code dodajemy katalog projektu, aby importować moduły z src/
    project_dir = os.getcwd()
    if project_dir not in sys.path:
        sys.path.insert(0, project_dir)

print("\n✅ Environment configured successfully! Ready to run the pipeline.")

In [ ]:
import cv2
import os
import glob
import matplotlib.pyplot as plt
import pandas as pd

# Import dedykowanych klas modułowych z folderu src
from src.wcag_analyzer import WCAGAnalyzer
from src.segmenter import KMeansSegmenter
from src.detector import UIDetector
from src.saliency_mapper import SaliencyMapper
from src.report_generator import ReportGenerator

# ==============================================================================
# 1. DYNAMICZNE MAPOWANIE KATALOGÓW WEJŚCIOWYCH I WYJŚCIOWYCH
# ==============================================================================
if IS_COLAB_CLOUD:
    dir_screenshots = f"{REPO_NAME}/data/screenshots"
    dir_icons = f"{REPO_NAME}/data/icons"
    dir_layouts = f"{REPO_NAME}/data/layouts"
    dir_output = f"{REPO_NAME}/output"
else:
    dir_screenshots = "data/screenshots"
    dir_icons = "data/icons"
    dir_layouts = "data/layouts"
    dir_output = "output"

# Tworzenie katalogu na wyniki
os.makedirs(dir_output, exist_ok=True)

# Funkcja pomocnicza do pobierania obrazów z rozszerzeniami PNG/JPG
def get_image_paths(folder_path):
    extensions = ["*.png", "*.jpg", "*.jpeg"]
    paths = []
    for ext in extensions:
        paths.extend(glob.glob(os.path.join(folder_path, ext)))
    return paths

# Pobieramy listy plików dla poszczególnych zadań
screenshot_files = get_image_paths(dir_screenshots)
icon_files = get_image_paths(dir_icons)
layout_files = get_image_paths(dir_layouts)

print(f"📊 Dataset Stats:")
print(f" |- Screenshots Found: {len(screenshot_files)}")
print(f" |- Icons/UI Elements Found: {len(icon_files)}")
print(f" |- Figma Layouts Found: {len(layout_files)}\n")

# Inicjalizacja centralnego generatora raportów
report = ReportGenerator(output_dir=dir_output)

# ==============================================================================
# 2. ANALIZA WCAG — DATASET REFERENCYJNY (wcag_colors.json)
# ==============================================================================
if IS_COLAB_CLOUD:
    wcag_colors_path = f"{REPO_NAME}/data/wcag_colors.json"
else:
    wcag_colors_path = "data/wcag_colors.json"

wcag_analyzer = WCAGAnalyzer(colors_path=wcag_colors_path)

print("====== ANALIZA WCAG — PARY KOLORÓW Z JSON ======")
df_wcag_reference = wcag_analyzer.analyze_all_pairs()
wcag_summary = wcag_analyzer.build_summary(df_wcag_reference)
wcag_recommendations = wcag_analyzer.generate_recommendations(df_wcag_reference)

# Rejestracja wyników WCAG w generatorze raportów
report.set_wcag_reference(df_wcag_reference, wcag_summary, wcag_recommendations)

display(df_wcag_reference[["name", "contrast_ratio", "grade", "recommendation"]])

print("\n📋 Rekomendacje WCAG:")
for rec in wcag_recommendations:
    print(rec)

# ==============================================================================
# 3. SEKWENCYJNE PRZETWARZANIE SYSTEMOWE (PĘTLA GŁÓWNA)
# ==============================================================================
print("\n====== STARTING BATCH PROCESSING ======")

# --- ETAP 1: Analiza interfejsów (Screenshots) -> Detekcja, Kontrast, Segmentacja ---
for idx, path in enumerate(screenshot_files):
    filename = os.path.basename(path)
    print(f"[{idx+1}/{len(screenshot_files)}] Processing screenshot: {filename}")
    
    try:
        # Inicjalizacja komponentów
        detector = UIDetector(min_area=250)
        segmenter = KMeansSegmenter(k_clusters=5)
        
        # Wykonanie algorytmów CV
        annotated_img, bboxes = detector.detect_elements(path)
        segmented_img = segmenter.segment(path)
        
        # Analiza kontrastu WCAG — pary z JSON + auto-próbkowanie kolorów ze screenshotu
        wcag_screenshot_df = wcag_analyzer.analyze_screenshot(path)
        wcag_metrics = wcag_analyzer.get_screenshot_metrics(wcag_screenshot_df)
        
        # Zapis grafik i raportu WCAG per screenshot
        name_clean = os.path.splitext(filename)[0]
        detected_path = os.path.join(dir_output, f"{name_clean}_detected.png")
        segmented_path = os.path.join(dir_output, f"{name_clean}_segmented.png")
        wcag_csv_path = wcag_analyzer.save_screenshot_report(
            wcag_screenshot_df, dir_output, filename
        )
        
        cv2.imwrite(detected_path, annotated_img)
        cv2.imwrite(segmented_path, segmented_img)
        
        # Rejestracja wyniku w generatorze raportów
        report.add_screenshot(
            filename=filename,
            detected_elements=len(bboxes),
            wcag_metrics=wcag_metrics,
            output_files={
                "detected": detected_path,
                "segmented": segmented_path,
                "wcag_csv": wcag_csv_path,
            },
        )
    except Exception as e:
        print(f" ❌ Error processing {filename}: {e}")

# --- ETAP 2: Generowanie map percepcji użytkownika (Figma Layouts) ---
for idx, path in enumerate(layout_files):
    filename = os.path.basename(path)
    print(f"[{idx+1}/{len(layout_files)}] Generating Saliency Map for layout: {filename}")
    
    try:
        mapper = SaliencyMapper()
        saliency_map, heatmap_overlay = mapper.generate_map(path)
        
        name_clean = os.path.splitext(filename)[0]
        saliency_path = os.path.join(dir_output, f"{name_clean}_saliency_heatmap.png")
        cv2.imwrite(saliency_path, heatmap_overlay)
        
        report.add_layout(
            filename=filename,
            output_files={"saliency": saliency_path},
        )
    except Exception as e:
        print(f" ❌ Error generating saliency for {filename}: {e}")

# ==============================================================================
# 4. GENEROWANIE RAPORTU KOŃCOWEGO
# ==============================================================================
print("\n====== GENEROWANIE RAPORTU KOŃCOWEGO ======")

generated_files = report.generate()

print("💾 Wygenerowane pliki raportu:")
for label, path in generated_files.items():
    print(f"  [{label}] {path}")

# Wyświetlenie zbiorczego raportu w notebooku
if report.screenshot_records or report.layout_records:
    df_final_report = pd.DataFrame(
        report.screenshot_records + report.layout_records
    )
    display(df_final_report)
else:
    print("⚠️ Brak przetworzonych danych. Dodaj obrazy do folderów data/.")